# Prepare Dynamic Wflow Handoff

Collect or reuse the shared Wflow warmup baseline, then prepare and QA the Wflow discharge time series consumed by SFINCS for one Event-Catalog event.

Stage Contract
- Requires: built Wflow submodels with SFINCS handoff gauges, AORC source access, and reviewed NHDPlus/STREAM-geo river geometry.
- Produces: shared Wflow warmup forcing, Wflow event forcing, dynamic Wflow `sfincs_discharge.nc`, dynamic handoff QA, and acceptance JSON.
- Next: run `c_run_example.ipynb` after the handoff status is accepted.


In [ ]:
import os
import warnings
from pathlib import Path

os.environ.pop("DEBUG", None)
os.environ.setdefault("MPLCONFIGDIR", "/tmp/flood-rm-matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")

import pandas as pd
import yaml
from IPython.display import display

# source inputs for Wflow warmup and event forcing.
from collect_sources import collect_warmup
# event selection, SFINCS staging, and handoff readiness.
from sfincs_runs.scenarios import plan_example
# standard paths and readiness tables.
from wflow_runs.notebook import load_runtime
# meteo forcing, discharge handoff, and acceptance gates.
from wflow_runs import (
    build_meteo,
    plan_warmup,
    prepare_instates,
    validate_geometry,
    validate_warmup_forcing,
    validate_instates,
)
from coupling.dynamic_handoff import plan_handoff, prepare_handoff
# model-physics check before dynamic handoff.
from wflow_runs.staticmaps_qa import validate_staticmaps
from wflow_runs.streamflow_realization import validate_wflow_streamflow_realization

runtime = load_runtime(Path("../..").resolve(), wflow_domain_review_required=False)
location_root = runtime.location_root
repo_root = runtime.repo_root
config = runtime.config
paths = runtime.paths


## 1 - Select Event and Plan Handoff


In [ ]:
EVENT_ID = None  # None -> highest-return-period catalog event; or e.g. "design_0398"
CATALOG_PATH = "data/event_catalog/catalog/probability_catalog.csv"

example_plan = plan_example(
    config,
    {"location_root": location_root},
    catalog_path=CATALOG_PATH,
    event_id=EVENT_ID,
)
display(example_plan)

if example_plan.event_id in (None, ""):
    raise RuntimeError(
        "No dynamic handoff event_id could be selected. "
        f"plan_example status={example_plan.status}; issues={example_plan.issues}"
    )

event_id = str(example_plan.event_id)
catalog = pd.read_csv(location_root / CATALOG_PATH, dtype={"event_id": str})
catalog["event_id"] = catalog["event_id"].astype(str).str.strip()
selected_rows = catalog[catalog["event_id"] == event_id]
if selected_rows.empty:
    preview = ", ".join(catalog["event_id"].head(8).astype(str))
    raise RuntimeError(f"Selected event_id {event_id!r} is not in {location_root / CATALOG_PATH}. First catalog IDs: {preview}")
selected = selected_rows.iloc[0]

handoff_plan = plan_handoff(config, location_root, event_id, catalog_path=location_root / CATALOG_PATH)
display(handoff_plan)

streamflow_realization = validate_wflow_streamflow_realization(
    config,
    location_root,
    event_id,
    catalog_path=location_root / CATALOG_PATH,
    event_model_root=None,
    raise_on_error=False,
)
display(streamflow_realization)
if streamflow_realization["status"].eq("failed").any():
    raise RuntimeError("Selected event is not ready for rainfall-driven Wflow generation (missing rainfall member or amplification reference gage).")


## Rerun Control


In [ ]:
rerun = True

## 2 - Verify Wflow Source Geometry and Static Maps


In [ ]:
river_geometry = location_root / config["collection"]["national_hydrography"]["river_geometry"]
display(validate_geometry(river_geometry, raise_on_error=False))

base_root = location_root / config["wflow"].get("base_model_root", "data/wflow/base")
domain_manifest = location_root / config["wflow"].get("domain_set_manifest", "data/wflow/domain_set.yaml")
configured_submodels = list(config["wflow"].get("domain_set", {}).get("submodels", []) or [])
if not configured_submodels and domain_manifest.exists():
    configured_submodels = list((yaml.safe_load(domain_manifest.read_text(encoding="utf-8")) or {}).get("submodels", []) or [])
if not configured_submodels:
    raise RuntimeError(f"No Wflow submodels found in config or generated manifest: {domain_manifest}")

qa_rows = []
for submodel in configured_submodels:
    model_root = base_root / str(submodel["wflow_submodel_id"])
    if (model_root / "staticmaps.nc").exists():
        report = validate_staticmaps(model_root, river_upa_km2=config["inland_coupling"]["discharge_forcing"].get("river_upa_km2"), raise_on_error=False)
        report.insert(0, "submodel_id", str(submodel["wflow_submodel_id"]))
        qa_rows.append(report)
    else:
        qa_rows.append(pd.DataFrame([{
            "submodel_id": str(submodel["wflow_submodel_id"]),
            "check": "staticmaps_exists",
            "status": "failed",
            "message": f"Missing staticmaps.nc: {model_root / 'staticmaps.nc'}",
        }]))
staticmap_qa = pd.concat(qa_rows, ignore_index=True) if qa_rows else pd.DataFrame()
display(staticmap_qa)
if not staticmap_qa.empty and staticmap_qa["status"].isin(["failed", "review_required"]).any():
    raise RuntimeError("Wflow staticmap QA must pass before dynamic SFINCS handoff.")


## 3 - Collect Shared Warmup AORC Forcing


In [ ]:
baseline_plan = plan_warmup(config, location_root)
display(baseline_plan)

collect_baseline_warmup = True
if collect_baseline_warmup:
    warmup_collection = collect_warmup(config, paths, force=rerun)
    display(pd.Series(warmup_collection, name="aorc_wflow_baseline_warmup"))

warmup_forcing = validate_warmup_forcing(
    config,
    location_root,
    event_id,
    reference_time=baseline_plan["baseline_reference_time"],
    raise_on_error=False,
)
display(warmup_forcing)
if warmup_forcing["status"].eq("failed").any():
    raise RuntimeError("Collect the shared AORC warmup baseline before running dynamic Wflow handoff.")

state_bootstrap = prepare_instates(config, location_root, force=rerun)
display(state_bootstrap)
if not state_bootstrap.empty and state_bootstrap["status"].eq("failed").any():
    raise RuntimeError("Create native Wflow instate/instates.nc before event replay.")

instate_readiness = validate_instates(config, location_root, raise_on_error=False)
display(instate_readiness)
if not instate_readiness.empty and instate_readiness["status"].eq("failed").any():
    raise RuntimeError("Create or promote native Wflow instate/instates.nc from the shared warmup baseline before event replay.")


## 4 - Stage Wflow Event Meteo


In [ ]:
meteo_report = build_meteo(
    config,
    location_root,
    event_id,
    catalog_path=location_root / CATALOG_PATH,
    overwrite=rerun,
)
display(pd.Series(meteo_report, name="wflow_event_meteo_forcing"))


## 5 - Run Dynamic Wflow Handoff QA


In [ ]:
run_dynamic_wflow = True

handoff_report = prepare_handoff(
    config,
    location_root,
    event_id,
    catalog_path=location_root / CATALOG_PATH,
    execute=run_dynamic_wflow,
)
display(handoff_report)

## 6 - Acceptance Artifact


In [ ]:
acceptance_path = location_root / handoff_plan["dynamic_handoff_acceptance"]
if acceptance_path.exists():
    display(pd.read_json(acceptance_path, typ="series"))
else:
    print("Dynamic handoff planned but not executed yet:", acceptance_path)
